In [ ]:
# Boilerplate imports

%load_ext autoreload
%autoreload 2
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

import numpy as np
np.set_printoptions(linewidth=5000)
import pandas as pd
pd.set_option("display.max_rows", 2000)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 2000)
from scipy.stats import norm
import scipy

from bokeh.plotting import figure
from bokeh.io import show, output_notebook, push_notebook
from bokeh.layouts import gridplot, row, column
from bokeh.models import (BasicTicker, ColorBar, ColumnDataSource, LinearColorMapper, PrintfTickFormatter, HoverTool)
from bokeh.transform import transform
from bokeh.models.formatters import DatetimeTickFormatter
output_notebook()

from functools import partial

import logging as Log
Log.basicConfig(level=Log.INFO)

verbose = False

In [ ]:
# Read data 

df_opt = pd.read_csv("optdata_JPM_noon_20210820_1500_20210723_12_00_00.csv")
df_opt['cmiv'] = np.where(df_opt['cbsz']==0, np.nan, df_opt['cmiv'].values)
df_opt['pmiv'] = np.where(df_opt['pbsz']==0, np.nan, df_opt['pmiv'].values)

df_und = pd.read_csv("unddata_JPM_noon_20210820_1500_20210723_12_00_00.csv")

display(df_und)
display(df_opt.head())

In [ ]:
# Generate mids and default option weights

fwd = df_und['f'].values[0]
vt = df_und['vt'].values[0]
Log.info("Using forward: %f voltime: %f", fwd, vt)

def call_weight(row):
    call_ok = 1 if row['cbiv'] > 0 else 0
    strike = row['kpx']    
    #call_weight = (1.0 if strike >= fwd else 0)  * call_ok
    call_weight = (1.0 - np.abs(row['cdelta'])) * call_ok
    return call_weight
    
def put_weight(row):

    put_ok =  1 if row['pbiv'] > 0 else 0
    strike = row['kpx']    
    #put_weight = (1.0 if strike <= fwd else 0)  * put_ok
    put_weight = (1.0 - np.abs(row['pdelta'])) * put_ok
    return put_weight    
    
def strike_target(row):
    call_weight = row['call_weight']
    put_weight = row['put_weight']
    target = (row['cmiv'] * call_weight + row['pmiv'] * put_weight) / (call_weight + put_weight) if call_weight + put_weight > 0 else np.nan
    return target

df_opt['call_weight'] = df_opt.apply(call_weight, axis=1)
df_opt['put_weight'] = df_opt.apply(put_weight, axis=1)
df_opt['tgt'] = df_opt.apply(strike_target, axis=1)
display(df_opt)

In [ ]:
# Plotting utility methods

def plot_fit(df, spl, strike_margin):
    source = ColumnDataSource(df)
    p = figure(plot_width=600, plot_height=400, tooltips=[('kpx', '$kpx'), ('cmiv', '$cmiv')])
    p.circle('kpx', 'cmiv', color='red',  fill_color="white", legend_label='cmiv', source=source)
    p.square('kpx', 'pmiv', color='blue', fill_color="white", legend_label='pmiv', source=source)
    p.circle(df.kpx[call_included], df.cmiv[call_included], color='red')
    p.square(df.kpx[put_included], df.pmiv[put_included], color='blue')
    klo = df.kpx.min() - strike_margin
    khi = df.kpx.max() + strike_margin
    strikes = np.linspace(klo, khi, 101)
    p.line(strikes, spl(strikes), color='green', legend_label='fit')
    p.legend.location = "top_left"
    p.legend.click_policy="hide"
    return p

def plot_residuals(df, spl, strike_margin):
    source = ColumnDataSource(df)
    theos = spl(df.kpx.values)
    p = figure(plot_width=600, plot_height=400, tooltips=[('kpx', '$kpx'), ('cmiv', '$cmiv')])
    p.circle(df.kpx, df.cmiv - theos, color='red',  fill_color="white", legend_label='cmiv')
    p.square(df.kpx, df.pmiv - theos, color='blue', fill_color="white", legend_label='pmiv',)
    p.circle(df.kpx[call_included], df.cmiv[call_included] - theos[call_included], color='red')
    p.square(df.kpx[put_included], df.pmiv[put_included] - theos[put_included], color='blue')
    klo = df.kpx.min() - strike_margin
    khi = df.kpx.max() + strike_margin
    strikes = np.linspace(klo, khi, 101)
    p.line(strikes, np.zeros_like(strikes), color='green', legend_label='fit')
    p.legend.location = "top_left"
    p.legend.click_policy="hide"
    return p

def plot_one_residual(p, theos, df, included, bidcol, askcol, color):
    # bids
    p.triangle(df.kpx, df[bidcol] - theos, color=color,  fill_color="white", legend_label=bidcol)
    p.line(df.kpx, df[bidcol] - theos, color=color, legend_label=bidcol)
    p.triangle(df.kpx[included], df[bidcol].values[included] - theos[included], color=color, legend_label=bidcol)
    # offers
    p.inverted_triangle(df.kpx, df[askcol] - theos, color=color,  fill_color="white", legend_label=askcol)
    p.line(df.kpx, df[askcol] - theos, color=color, legend_label=askcol)
    p.inverted_triangle(df.kpx[included], df[askcol].values[included] - theos[included], color=color, legend_label=askcol)    

def plot_residuals_bid_ask(df, spl, strike_margin):
    source = ColumnDataSource(df)
    theos = spl(df.kpx.values)
    p = figure(plot_width=600, plot_height=400, tooltips=[('kpx', '$kpx'), ('cmiv', '$cmiv')])
    # calls
    plot_one_residual(p, theos, df, call_included, 'cbiv', 'caiv', 'red')
    # puts
    plot_one_residual(p, theos, df, put_included, 'pbiv', 'paiv', 'blue')

    klo = df.kpx.min() - strike_margin
    khi = df.kpx.max() + strike_margin
    p.line([klo, khi], [0, 0], color='green', legend_label='fit')
    p.legend.location = "top_left"
    p.legend.click_policy="hide"
    return p

def d12(F, K, stddev):
    d1 = np.log(F/K)/stddev + stddev/2.0
    d2 = d1 - stddev
    return d1, d2

def blackPrice(cp, F, K, df, sigma, T):
    sqrtT = np.sqrt(T)
    stddev = sigma * sqrtT
    d1, d2 = d12(F, K, stddev)
    price = cp * df * ( F * norm.cdf(cp*d1) - K * norm.cdf(cp*d2) )    
    return price

def bsprice(S, K, vol, cp, T, volT, box, roll):
    F = S*np.exp(roll*T)
    df = np.exp(-box*T) 
    return blackPrice(cp, F, K, df, vol, volT)


def plot_implied_pdf(df, fwd, vt, spl, strike_margin):
    p = figure(plot_width=600, plot_height=400)
    klo = df.kpx.min() - strike_margin
    khi = df.kpx.max() + strike_margin
    strikes = np.linspace(klo, khi, 1000)
    strikevols = spl(strikes)
    call_prices = [blackPrice(1, fwd, K, 1.0, sigma, vt) for K, sigma in zip(strikes, strikevols)]
    butterflies = np.diff(call_prices, 2, prepend=[np.nan], append=[np.nan])
    p.line(strikes, butterflies, color='green', legend_label='pdf')
    p.legend.location = "top_left"
    p.legend.click_policy="hide"
    return p

In [ ]:
# Prepare data to fit

df_to_fit = df_opt
vt = df_und['vt'].values[0]
fwd = df_und['f'].values[0]

tslm0 = np.log(df_to_fit['kpx'].values / fwd) / np.sqrt(vt)
iv0 = np.array(df_to_fit['tgt'].values) 

call_included = df_to_fit['call_weight'] > 0
put_included = df_to_fit['put_weight'] > 0

In [ ]:
# fit a smoothing spline
from scipy.interpolate import UnivariateSpline

# smoothing parameter
s = 0.4 / len(df_opt)  

#degree of spline
k = 3

# extra strike margin to plot
strike_margin = 10

x = df_to_fit.kpx.values
y = np.array(df_to_fit.tgt.values)  # make a copy b/c we'll set values
ok = np.isfinite(df_to_fit.tgt)
y[~ok] = 0 
w = np.where(ok, 1.0, 0.0) * (df_opt.pvega.values + df_opt.cvega.values)

if verbose:
    Log.info("x  %s", x)
    Log.info("y  %s", y)
    Log.info("w  %s", w)

spl = UnivariateSpline(x, y, w=w, s=s, k=k)

p_curve = plot_fit(df_to_fit, spl, strike_margin)
p_implied_pdf = plot_implied_pdf(df_to_fit, fwd, vt, spl, strike_margin)
p_residuals = plot_residuals(df_to_fit, spl, strike_margin)
p_residuals_bid_ask = plot_residuals_bid_ask(df_to_fit, spl, strike_margin)

show(gridplot([[p_curve, p_residuals], [p_implied_pdf, p_residuals_bid_ask]]))

Log.info("Knots: %s", spl.get_knots())
Log.info("Coeffs: %s", spl.get_coeffs())

In [ ]:
# S3 fit

def S3(ns, vol, s2, c2):
    t = 1 + s2*ns
    return vol * np.sqrt(0.5*t + 0.5*np.sqrt(t*t + 2*c2*ns*ns))

def S3tslm(tslm, vol, s2, c2):
    ns = tslm / vol
    return S3(ns, vol, s2, c2)

def cost_S3(vol, s2, c2, tslm, iv, wgt):
    prediction = S3tslm(tslm, vol, s2, c2) 
    return np.sum(wgt * np.square(prediction - iv))
    
# A couple of weight schemes...
    
# constant weights
wgt0 = np.ones_like(tslm0)

# ... or emphasize near-ATM
min_tslm = -0.3
max_tslm = 0.5
wgt1 = np.where(np.logical_and(tslm0 >= min_tslm, tslm0 < max_tslm), 1, 0) * 1/(np.square(tslm0) + 0.0001) * (1/iv0)**5

# filter out NaN's
idx = np.where(np.isfinite(iv0))[0]
tslm = tslm0[idx]
iv = iv0[idx]
wgt = wgt1[idx]  # select weight scheme here
    
if verbose:
    Log.info("tslm: %s", tslm)
    Log.info("iv: %s", iv)
    Log.info("wgt: %s", wgt)
    
def objective_fn(x):
    return cost_S3(x[0], x[1], x[2], tslm, iv, wgt)    
    
# optimize
result = scipy.optimize.minimize(objective_fn, x0=[0.3, 0, 0], tol=1e-8)

if verbose:
    Log.info("result:\n%s", result)
fitpars = result.x
Log.info("Fit params: %s", fitpars.tolist())

# vol curve as a function of strike
def S3_fitted_result(kpx):
    tslm = np.log(kpx/fwd) / np.sqrt(vt)
    return S3tslm(tslm, *fitpars)              

p_curve = plot_fit(df_to_fit, S3_fitted_result, strike_margin)
p_residuals = plot_residuals(df_to_fit, S3_fitted_result, strike_margin)
p_implied_pdf = plot_implied_pdf(df_to_fit, fwd, vt, S3_fitted_result, strike_margin)
p_residuals_bid_ask = plot_residuals_bid_ask(df_to_fit, S3_fitted_result, strike_margin)

show(gridplot([[p_curve, p_residuals], [p_implied_pdf, p_residuals_bid_ask]]))